In [35]:
# Code adapted from:
# 1. https://huggingface.co/docs/transformers/en/training
# 2. https://huggingface.co/docs/transformers/en/tasks/sequence_classification

In [36]:
!pwd

/Users/ryohskay/Documents/uni/peshitta_cj/cite_tf


In [37]:
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
# from torch.nn import LayerNorm
# from torch.optim import AdamW, lr_scheduler
# from transformers.trainer_pt_utils import get_parameter_names
from tf_keras.optimizers import AdamW
from tf_keras.preprocessing import text_dataset_from_directory
from datasets import load_dataset
from pathlib import Path
from tf_keras import metrics
from tensorflow import cast, float32
import numpy as np
import evaluate

In [38]:
dataset_dir = Path("./out/ETCBC_Peshitta")

In [39]:
batch_size = 32
seed = 42

raw_train_ds = text_dataset_from_directory(
    str(dataset_dir / "train"),
    batch_size=batch_size,
    # validation_split=0.2,
    # subset='training',
    shuffle=False,
    seed=seed)

raw_test_ds = text_dataset_from_directory(
    str(dataset_dir / "test"),
    batch_size=batch_size)

Found 5841 files belonging to 2 classes.
Found 1562 files belonging to 2 classes.


In [40]:
for text_batch, label_batch in raw_train_ds.take(1):
  for i in range(3):
    print("Review", text_batch.numpy()[i])
    print("Label", label_batch.numpy()[i])

Review b'WHLJN CM"H> DBN"J >JSRJL D<LW LMYRJN <M J<QWB GBR WBJTH <LW'
Label 0
Review b'RWBJL WCM<WN WLWJ WJHWD>'
Label 0
Review b'W>JSKR WZBWLWN WBNJMJN'
Label 0


In [41]:
print("Label 0 corresponds to", raw_train_ds.class_names[0])
print("Label 1 corresponds to", raw_train_ds.class_names[1])

Label 0 corresponds to Jewish
Label 1 corresponds to Xaristos


In [42]:
# Load data from json files into Huggingface's Dataset class object

# dataset = load_dataset("json", data_files={"train": ["./out/ot_train.json", "./out/nt_train.json"], "validation": ["./out/ot_test.json", "./out/nt_test.json"]})

In [43]:
# print(dataset["train"][0])

In [44]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilroberta-base")

In [59]:
def tokenize_function(data):
    return tokenizer(
        data["text"],
    )

In [63]:
tokenized_data = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=4,
)

Map (num_proc=4):   0%|          | 0/5841 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1562 [00:00<?, ? examples/s]

In [47]:
# print(tokenized_data)

In [48]:
# id2label = {0: "Jewish", 1: "Christian"}
# label2id = {"Jewish": 0, "Christian": 1}

In [49]:
model = TFAutoModelForSequenceClassification.from_pretrained("distilbert/distilroberta-base", num_labels=2)

All PyTorch model weights were used when initializing TFRobertaForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFRobertaForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [52]:
# tf_dataset = model.prepare_tf_dataset(tokenized_data["train"], batch_size=16, shuffle=True, tokenizer=tokenizer)

In [62]:
# train_ds = raw_train_ds.map(tokenizer)

ValueError: in user code:

    File "/Users/ryohskay/.pyenv/versions/3.12.8/lib/python3.12/site-packages/transformers/tokenization_utils_base.py", line 2887, in __call__  *
        encodings = self._call_one(text=text, text_pair=text_pair, **all_kwargs)
    File "/Users/ryohskay/.pyenv/versions/3.12.8/lib/python3.12/site-packages/transformers/tokenization_utils_base.py", line 2947, in _call_one  *
        raise ValueError(

    ValueError: text input must be of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).


In [53]:
# metric = evaluate.load("accuracy")

# def compute_metrics(eval_pred):
#     logits, labels = eval_pred
#     predictions = np.argmax(logits, axis=-1)
#     return metric.compute(predictions=predictions, references=labels)

In [56]:
model.compile(
    optimizer=AdamW(learning_rate=2e-05, epsilon=1e-08),
    metrics=[
        metrics.F1Score(),
        metrics.AUC(),
    ],
)
model.fit()

ValueError: text input must be of type `str` (single example), `List[str]` (batch or single pretokenized example) or `List[List[str]]` (batch of pretokenized examples).

In [57]:
tf_test = model.prepare_tf_dataset(tokenized_data["validation"], batch_size=16, shuffle=True, tokenizer=tokenizer)

In [60]:
loss = model.evaluate(tf_test, verbose=1)

97/97 [==============================] - 52s 530ms/step - loss: 0.2832


0.28319957852363586

In [62]:
model.compute_metrics(tf_test)

TypeError: Model.compute_metrics() missing 3 required positional arguments: 'y', 'y_pred', and 'sample_weight'

In [45]:
tokenized_data["validation"][0]["label"]

0

In [2]:
inputs = tokenizer(dataset["train"][0]["text"], padding="max_length", truncation=True, return_tensors="pt")
outputs = model(**inputs)

NameError: name 'tokenizer' is not defined

In [159]:
# sentencepiece

In [ ]:
# serious re-implementation for real data